# Design constraints report (appendix)

Classification metrics of the 3 designs on the frozen test set, plus the design-constraint and trade-off results.

- **Computed here:** accuracy, precision, recall, F1 and confusion matrices. One pass over `dataset/test`.
- **Read from the CSVs, not measured here:** inference time, memory, Maintainability Index (`evaluate.py`, 20 fresh-process runs on the MiniPC) and training time (`train.py`, Colab T4). Colab and other PCs have a different CPU and RAM, so never measure speed or memory in this notebook.

**Colab:** put `dataset.zip`, `designs_out.zip` and (after `evaluate.py` + `tradeoff.py` on the MiniPC) `results.zip` in `MyDrive/cbp/`. Make `results.zip` on the MiniPC from the repo folder:
`python -c "import shutil; shutil.make_archive('results', 'zip', '.', 'evaluation/results')"`

**Local:** start Jupyter from the repo folder (or `evaluation/`), with `dataset/test` and `models/d*.pth` in place.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    BRANCH = "feat/pipeline-and-setup"
    D = "/content/drive/MyDrive/cbp"
    from google.colab import drive
    drive.mount("/content/drive")
    !git clone -q -b $BRANCH https://github.com/codeyson/cbp-crack-detection.git
    %cd cbp-crack-detection
    !pip install -q -r evaluation/requirements.txt
    for z in ("dataset.zip", "designs_out.zip", "results.zip"):   # results.zip is optional
        if os.path.exists(f"{D}/{z}"):
            !unzip -q -o "{D}/{z}" -d .
elif os.path.basename(os.getcwd()) == "evaluation":
    os.chdir("..")

import glob
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
sys.path.insert(0, "evaluation")
import designs, tradeoff
from designs import DESIGNS

R, GRADES = "evaluation/results", ["A", "B", "C"]
NAMES = {d: f"{d.upper()} {m}" for d, m in DESIGNS.items()}
print("weights:", sorted(glob.glob("models/d*.pth")))
print("results:", sorted(os.listdir(R)) if os.path.isdir(R) else "none yet")

## Test set and predictions

Each design predicts every test image once (CPU). Image reading and preprocessing are the same as in `evaluate.py`.

In [ ]:
items = [(p, os.path.basename(os.path.dirname(p))) for p in sorted(glob.glob("dataset/test/*/*"))]
y_true = [g for _, g in items]
assert items, "dataset/test is empty"
preds = {}
for d in DESIGNS:
    predict = designs.load(d, designs.weights_path(d))
    preds[d] = [predict(cv2.imread(p)) for p, _ in items]
pd.Series(y_true).value_counts().sort_index().rename_axis("grade").to_frame("test images")

## Classification report

Per grade: precision, recall, F1-score and support (number of test images).

In [ ]:
for d in DESIGNS:
    print(f"=== {NAMES[d]} ===   accuracy {accuracy_score(y_true, preds[d]):.4f}")
    print(classification_report(y_true, preds[d], labels=GRADES, digits=4, zero_division=0))

## Confusion matrices

Rows are the true grade, columns the predicted grade.

In [ ]:
fig, ax = plt.subplots(1, len(DESIGNS), figsize=(5 * len(DESIGNS), 4.5))
for a, d in zip(ax, DESIGNS):
    ConfusionMatrixDisplay(confusion_matrix(y_true, preds[d], labels=GRADES), display_labels=GRADES).plot(
        ax=a, cmap="Blues", colorbar=False)
    a.set_title(NAMES[d])
plt.tight_layout()
plt.show()

## Classification metrics summary

Macro average: each grade counts equally, so a weak grade C pulls it down even though C has fewer test images.

In [ ]:
rows = []
for d in DESIGNS:
    r = classification_report(y_true, preds[d], labels=GRADES, output_dict=True, zero_division=0)
    rows.append(dict(design=NAMES[d], accuracy=r["accuracy"], misclassification_pct=(1 - r["accuracy"]) * 100,
                     macro_precision=r["macro avg"]["precision"], macro_recall=r["macro avg"]["recall"],
                     macro_f1=r["macro avg"]["f1-score"],
                     **{f"recall_{g}": r[g]["recall"] for g in GRADES}, **{f"F1_{g}": r[g]["f1-score"] for g in GRADES}))
pd.DataFrame(rows).set_index("design").round(4)

## Design constraints

Mean of the newest `evaluate.py` batch per design (same values `tradeoff.py` uses) and the training time from `train.py`. `-` = not measured yet.

In [ ]:
cons = tradeoff.load_constraints()
series = tradeoff.gather(cons, R)
table = pd.DataFrame({NAMES.get(d, d): {f"{c['name']}: {c['metric']} ({c['better']} is better)":
                                        (sum(s[c["name"]]) / len(s[c["name"]]) if s[c["name"]] else None) for c in cons}
                      | {"runs averaged": max(len(v) for v in s.values())} for d, s in series.items()})
table.round(5).fillna("-")

## Trade-off analysis

Scores, overall ranking, Pareto and sensitivity from `tradeoff.py`, with its charts.

In [ ]:
tradeoff.run(R)
for f in ("radar.png", "sensitivity.png", "runs.png"):
    if os.path.exists(f"{R}/{f}"):
        display(Image(f"{R}/{f}", width=700))